# Mathematical Analysis of Thermal Performance in Ventilated Facades

## 1. Introduction
In modern construction, ventilated facades are essential for energy efficiency and structural longevity. However, the continuity of insulation is often interrupted by structural components like aluminum brackets and anchors. These intersections create "thermal bridges."

## 2. Problem Formulation
The goal of this project is to mathematically model the heat transfer through a multi-layered ventilated facade. We will:
* Calculate the total thermal resistance ($R_{total}$) of a wall assembly.
* Determine the overall Heat Transfer Coefficient ($U$-value).
* Analyze the impact of point thermal bridges using mathematical approximations.

## 3. Mathematical Background

### 3.1. Heat Transfer Physics
To analyze the U-value, we rely on Fourier's Law of Heat Conduction. In a one-dimensional steady state, the rate of heat transfer is defined by the derivative of temperature ($T$) with respect to distance ($x$):
$$q = -k \frac{dT}{dx}$$
This derivative represents the temperature gradient across the facade layers.

### 3.2. Modelling Multi-layered Assemblies
We treat the facade as a system of thermal resistances. For a wall with $n$ layers, the total thermal resistance ($R_{tot}$) is calculated as:
$$R_{tot} = R_{si} + \sum_{i=1}^{n} \frac{d_i}{\lambda_i} + R_{se}$$
Where:
* $d_i$ is the thickness of layer $i$.
* $\lambda_i$ is the thermal conductivity of the material.

The Heat Transfer Coefficient ($U$-value) is then:
$$U = \frac{1}{R_{tot}}$$

### 3.3. Manual Calculation Example
To verify the logic of our future Python implementation, we will calculate the U-value for a high-performance ventilated facade assembly.

| No. | Layer | Thickness ($d$) [m] | Conductivity ($\lambda$) [W/mK] | Resistance ($R = d/\lambda$) |
| :--- | :--- | :--- | :--- | :--- |
| - | Internal Surface ($R_{si}$) | - | - | 0.130 |
| 1 | Drywall | 0.015 | 0.250 | 0.060 |
| 2 | Mineral Wool (in-stud) | 0.100 | 0.040 | 2.500 |
| 3 | Sheathing Board | 0.012 | 0.170 | 0.071 |
| 4 | Rockwool (external) | 0.150 | 0.035 | 4.286 |
| 5 | Air Gap (ventilated) | 0.050 | - | 0.130* |
| 6 | Fibre Cement Panel | 0.008 | 1.200 | 0.007 |
| - | External Surface ($R_{se}$) | - | - | 0.040 |

*\*Note: For a well-ventilated air gap, the thermal resistance is often taken as the sum of surface resistances, but here we use a standard simplified R-value for air layers.*

**Total Resistance ($R_{tot}$):**
$$R_{tot} = 0.130 + 0.060 + 2.500 + 0.071 + 4.286 + 0.130 + 0.007 + 0.040 = 7.224 \, m^2K/W$$

**Total U-value:**
$$U = \frac{1}{7.224} \approx 0.138 \, W/m^2K$$

### 3.4. Python Implementation
In this section, we translate the mathematical model into a functional Python program. By using a list of dictionaries to represent material layers, we create a flexible tool that can handle any number of facade components.

In [7]:
# Material Database for the wall build-up
# d = thickness [m], l = lambda (thermal conductivity) [W/mK]
build_up = [
    {"name": "Drywall", "d": 0.015, "l": 0.250},
    {"name": "Mineral Wool (in SFS)", "d": 0.100, "l": 0.040},
    {"name": "Sheathing Board", "d": 0.012, "l": 0.170},
    {"name": "Rockwool", "d": 0.150, "l": 0.035},
    {"name": "Air Gap", "d": 0.050, "l": 0.385}, # R=0.13 is approx l=0.385 at 50mm
    {"name": "Fibre Cement Panel", "d": 0.008, "l": 1.200}
]

def calculate_u_value(layers, r_si=0.13, r_se=0.04):
    """
    Calculates the total thermal resistance and U-value.
    """
    total_r = r_si + r_se
    
    print(f"{'Material':<25} | {'R-value (m2K/W)':<20}")
    print("-" * 45)
    
    for layer in layers:
        r_layer = layer["d"] / layer["l"]
        total_r += r_layer
        print(f"{layer['name']:<25} | {r_layer:.3f}")
        
    u_value = 1 / total_r
    return total_r, u_value


total_resistance, u_result = calculate_u_value(build_up)

print("-" * 45)
print(f"Total Thermal Resistance: {total_resistance:.3f} m2K/W")
print(f"Final U-value: {u_result:.3f} W/m2K")

Material                  | R-value (m2K/W)     
---------------------------------------------
Drywall                   | 0.060
Mineral Wool (in SFS)     | 2.500
Sheathing Board           | 0.071
Rockwool                  | 4.286
Air Gap                   | 0.130
Fibre Cement Panel        | 0.007
---------------------------------------------
Total Thermal Resistance: 7.223 m2K/W
Final U-value: 0.138 W/m2K


## 4. Structural Thermal Bridges & Model Refinement

### 4.1. Accounting for Heat Loss via Brackets
In ventilated facades, aluminum brackets penetrate the insulation layer to support the external panels. These act as "thermal bridges," significantly increasing the heat flow. 

According to the Scientific Method, we must refine our initial model to account for these "errors." We calculate the Effective U-value ($U_{eff}$) by adding the point thermal bridge impact ($\chi$) to our base calculation.

**The Correction Formula:**
$$U_{eff} = U_{clear} + \Delta U$$
$$\Delta U = \frac{\sum_{i=1}^{n} \chi_i}{A}$$

**Simplified Form for Identical Brackets:**
When all brackets in the assembly are identical, the formula simplifies to:
$$U_{eff} = U_{clear} + \frac{n \cdot \chi}{A}$$

**Where:**
* $U_{eff}$ is the effective (real-world) U-value ($W/m^2K$).
* $U_{clear}$ is the Clear Wall U-value ($W/m^2K$).
* $\Delta U$ is the additional heat loss due to thermal bridging ($W/m^2K$).
* $n$ is the total number of brackets.
* $\chi$ (Chi) is the point thermal bridge coefficient ($W/K$).
* $A$ is the total area of the facade ($m^2$).

In [8]:
# Constants for the Thermal Bridge Analysis
chi_bracket = 0.015  # W/K (typical value for an aluminum bracket with thermal pad)
brackets_per_m2 = 2.5 # Average density for a standard facade grid

# Calculate the impact (Delta U)
delta_u = brackets_per_m2 * chi_bracket

# Add impact result to Clear Wall U-value result
u_effective = u_result + delta_u

print(f"Clear Wall U-value: {u_result:.3f} W/m2K")
print(f"Correction for Thermal Bridges (Delta U): {delta_u:.3f} W/m2K")
print(f"Effective U-value: {u_effective:.3f} W/m2K")

# Error Analysis: Calculate the percentage increase
increase = (delta_u / u_result) * 100
print(f"The thermal bridges increase heat loss by: {increase:.2f}%")

Clear Wall U-value: 0.138 W/m2K
Correction for Thermal Bridges (Delta U): 0.037 W/m2K
Effective U-value: 0.176 W/m2K
The thermal bridges increase heat loss by: 27.09%
